# 01 - Colab Environment Setup (Phase 0)

**Robust One-Shot Instance Segmentation Under Viewpoint, Illumination, Scale
and Occlusion Changes.**

This notebook builds and verifies a reproducible Google Colab environment for
the whole project. It is **Phase 0 only**: it sets up storage, code, seeding and
the feature-cache contract. It deliberately does **not** download datasets,
download DINOv2/SAM weights, or implement Matcher / PerSAM / the proposed
method.

| Section | Purpose |
| --- | --- |
| 1 | Project configuration (the only cell you normally edit) |
| 2 | Google Drive mount (persistent storage) |
| 3 | Repository clone / update |
| 4 | Python dependencies (Phase 0 set only) |
| 5 | GPU diagnostics + CUDA inference test |
| 6 | Project directory tree |
| 7 | Reproducibility: seeds, determinism, experiment metadata |
| 8 | Feature-cache architecture (dummy tensors only) |
| 9 | End-to-end environment verification |
| 10 | Next steps and phase boundaries |

**The notebook is safe to re-run.** Re-running skips clones, package installs
that are already satisfied, existing directories and existing cache entries.

> Before the first run: set a **GPU** runtime
> (`Runtime > Change runtime type > Hardware accelerator > GPU`).

## Section 1 - Project configuration

Everything is driven from the variables below. Change them here and re-run;
nothing else in the notebook needs editing.

* `GITHUB_USER` / `GITHUB_REPO` - the repository to clone into `/content`.
* `DRIVE_FOLDER` - folder inside `MyDrive` that holds all persistent data.
* `USE_LOCAL_REPO` - set `True` if you uploaded the repository to the runtime
  some other way (zip, Drive, Colab file upload).
* `INSTALL_*` switches keep the Phase 0 install minimal on purpose. Matcher,
  SAM/SAM2 and PerSAM dependencies are added when those phases begin.

In [ ]:
# ============================== EDIT HERE ==============================
GITHUB_USER       = "BellilxDhaker"              # <- your GitHub username
GITHUB_REPO       = "cvision"                    # <- repository name
GITHUB_BRANCH     = "main"                      # <- branch to check out
DRIVE_FOLDER      = "one-shot-segmentation"      # folder inside MyDrive

MOUNT_DRIVE       = True    # mount Google Drive (persistent storage)
USE_LOCAL_REPO    = False   # True if the repo is already in the runtime
UPDATE_REPO       = False   # True -> `git pull --ff-only` an existing clone
USE_GITHUB_TOKEN  = False   # True -> read GITHUB_TOKEN from Colab secrets

INSTALL_PACKAGES  = True    # Phase 0 python packages
INSTALL_TORCH     = False   # Colab already ships a CUDA-matched torch
INSTALL_TORCHVISION = True
INSTALL_TRANSFORMERS = True # DINOv2 backbone (weights are NOT downloaded yet)
INSTALL_DEMO_UI   = False   # gradio, only for the demo phase

RANDOM_SEED       = 1234
DETERMINISTIC     = True
# ==========================================================================

import os
import textwrap

PROJECT_NAME = "robust-one-shot-segmentation"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# Canonical locations. Section 3 loads configs/paths.yaml, which is the single
# source of truth; these values document the expected layout and are cross-checked
# in Section 9.
PROJECT_ROOT     = "/content/one-shot-segmentation"
DRIVE_MOUNT      = "/content/drive"
DRIVE_ROOT       = f"{DRIVE_MOUNT}/MyDrive/{DRIVE_FOLDER}"
DATA_ROOT        = f"{DRIVE_ROOT}/data"
RAW_ROOT         = f"{DATA_ROOT}/raw"
PROCESSED_ROOT   = f"{DATA_ROOT}/processed"
CACHE_ROOT       = f"{DATA_ROOT}/cache"
SPLITS_ROOT      = f"{DATA_ROOT}/splits"
CHECKPOINT_ROOT  = f"{DRIVE_ROOT}/checkpoints"
RESULTS_ROOT     = f"{DRIVE_ROOT}/results"
LOGS_ROOT        = f"{DRIVE_ROOT}/logs"

CONFIG = {
    "GITHUB_REPO": REPO_URL,
    "GITHUB_BRANCH": GITHUB_BRANCH,
    "DRIVE_FOLDER": DRIVE_FOLDER,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "PROJECT_ROOT": PROJECT_ROOT,
    "DRIVE_ROOT": DRIVE_ROOT,
    "DATA_ROOT": DATA_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "RESULTS_ROOT": RESULTS_ROOT,
    "CACHE_ROOT": CACHE_ROOT,
    "RANDOM_SEED": RANDOM_SEED,
    "DETERMINISTIC": DETERMINISTIC,
}

# Keep it visible which run produced the results directory.
print(textwrap.dedent(f"""
    project      : {PROJECT_NAME}
    repository   : {REPO_URL} ({GITHUB_BRANCH})
    project root : {PROJECT_ROOT}
    drive root   : {DRIVE_ROOT}
    data         : {DATA_ROOT}
    checkpoints  : {CHECKPOINT_ROOT}
    results      : {RESULTS_ROOT}
    seed         : {RANDOM_SEED} (deterministic={DETERMINISTIC})
""").strip())

if "YOUR_GITHUB_USERNAME" in GITHUB_USER:
    print("! Set GITHUB_USER/GITHUB_REPO in Section 1, or set USE_LOCAL_REPO = True.")

## Section 2 - Google Drive

`/content` is **ephemeral**: it is wiped every time Colab recycles a runtime, so
datasets, checkpoints, cached features and results must live on Drive.

The cell mounts Drive only if it is not already mounted, and reports free
space. If mounting is not possible (for example outside Colab), the persistent
tree falls back to a local folder and the notebook prints a warning - use the
Drive version for real experiments.

In [ ]:
import os

def _mount_drive(mount_point: str = DRIVE_MOUNT) -> dict:
    """Mount Google Drive once. Returns a status dict (no exception on failure)."""
    mydrive = os.path.join(mount_point, "MyDrive")
    if os.path.isdir(mydrive):
        return {"available": True, "mounted_now": False,
                "message": f"Already mounted at {mount_point}."}
    try:
        from google.colab import drive  # noqa: F401
    except Exception as exc:
        return {"available": False, "mounted_now": False,
                "message": f"google.colab unavailable ({exc})."}
    try:
        drive.mount(mount_point, force_remount=False, timeout_ms=300_000)
        return {"available": os.path.isdir(mydrive), "mounted_now": True,
                "message": f"Mounted Google Drive at {mount_point}."}
    except Exception as exc:
        return {"available": False, "mounted_now": False,
                "message": f"Mount failed: {type(exc).__name__}: {exc}"}


drive_status = {"available": False, "message": "skipped (MOUNT_DRIVE = False)"}
if MOUNT_DRIVE:
    drive_status = _mount_drive(DRIVE_MOUNT)

PERSISTENT_ROOT = DRIVE_ROOT if drive_status["available"] else f"/content/{DRIVE_FOLDER}"
os.makedirs(PERSISTENT_ROOT, exist_ok=True)

if drive_status["available"]:
    import shutil
    free_gb = shutil.disk_usage(PERSISTENT_ROOT).free / 1024**3
    print(f"[drive] {drive_status['message']}")
    print(f"[drive] persistent root : {PERSISTENT_ROOT}")
    print(f"[drive] free space       : {free_gb:.1f} GB")
    if free_gb < 20:
        print("! Less than 20 GB free on Drive: datasets and cached features will not fit.")
else:
    print(f"[drive] {drive_status['message']}")
    print(f"[drive] falling back to a local (ephemeral) root: {PERSISTENT_ROOT}")
    print("! Results and caches will be LOST when the runtime is recycled.")

## Section 3 - Repository

The project repository is cloned into `/content/one-shot-segmentation`. The
repo holds **code and configuration only**; large artefacts stay on Drive.

If the clone already exists the step is skipped (set `UPDATE_REPO = True` in
Section 1 to fast-forward it). Afterwards the repository root is added to
`sys.path`, so `scripts.*` and the future `models` / `matching` / ... packages
are importable from any later cell.

In [ ]:
import importlib
import os
import sys


def _github_token() -> str:
    """Optional token from Colab secrets; empty string when unavailable."""
    if not USE_GITHUB_TOKEN:
        return ""
    try:
        from google.colab import userdata  # noqa: PLC0415

        return userdata.get("GITHUB_TOKEN") or ""
    except Exception as exc:
        print(f"[repo] token unavailable ({exc}); using anonymous clone.")
        return ""


def _git(*args, cwd=None):
    import subprocess

    out = subprocess.run(["git", *args], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, check=False)
    return out.returncode, out.stdout.decode("utf-8", errors="replace").strip()


if USE_LOCAL_REPO or os.path.isdir(os.path.join(PROJECT_ROOT, ".git")):
    repo_action = "local" if USE_LOCAL_REPO else "already_present"
    print(f"[repo] using existing checkout at {PROJECT_ROOT} ({repo_action})")
else:
    os.makedirs(os.path.dirname(PROJECT_ROOT), exist_ok=True)
    token = _github_token()
    url = REPO_URL
    if token:
        url = REPO_URL.replace("https://github.com/", f"https://{token}@github.com/")
    print(f"[repo] cloning {GITHUB_USER}/{GITHUB_REPO} ({GITHUB_BRANCH}) -> {PROJECT_ROOT}")
    code_out, output = _git("clone", "--branch", GITHUB_BRANCH, "--depth", "1", url, PROJECT_ROOT)
    if code_out != 0:
        print(output)
        print("! Clone failed. Check the repository name/visibility in Section 1, "
              "or set USE_LOCAL_REPO = True if the repo is private and you "
              "uploaded it manually.")
    else:
        if token:  # never keep the token in the remote URL
            _git("remote", "set-url", "origin", REPO_URL, cwd=PROJECT_ROOT)
        print(f"[repo] cloned at {GITHUB_BRANCH}")

if UPDATE_REPO and os.path.isdir(os.path.join(PROJECT_ROOT, ".git")):
    _code, output = _git("pull", "--ff-only", "origin", GITHUB_BRANCH, cwd=PROJECT_ROOT)
    print(f"[repo] pull: {output.splitlines()[-1] if output else 'nothing to do'}")

# Importable project packages.
if os.path.isdir(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"[repo] sys.path entry added: {PROJECT_ROOT in sys.path}")

REQUIRED_FILES = [
    "configs/paths.yaml",
    "scripts/environment_check.py",
    "scripts/reproducibility.py",
    "scripts/cache_utils.py",
    "scripts/project_setup.py",
]
missing = [f for f in REQUIRED_FILES if not os.path.isfile(os.path.join(PROJECT_ROOT, f))]
if missing:
    print("! Missing expected project files: " + ", ".join(missing))
else:
    print(f"[repo] all {len(REQUIRED_FILES)} Phase 0 project files present")

_git_head_code, git_head = _git("rev-parse", "--short", "HEAD", cwd=PROJECT_ROOT)
_git_branch_code, git_branch = _git("rev-parse", "--abbrev-ref", "HEAD", cwd=PROJECT_ROOT)
if _git_head_code == 0 and _git_branch_code == 0:
    print(f"[repo] git: branch={git_branch} commit={git_head}")
else:
    print("[repo] git: no commit information (not a git checkout - fine for "
          "USE_LOCAL_REPO, but experiment metadata will not record a commit)")

## Section 4 - Python dependencies (Phase 0 set only)

Installed packages are the ones this phase genuinely needs:

* **Core:** `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `pillow`, `tqdm`,
  `pandas`, `pyyaml`, `opencv-python-headless`
* **Vision:** `torchvision`, `transformers` (DINOv2 backbone)
* **PyTorch:** only when `INSTALL_TORCH = True`. Colab already ships a
  CUDA-matched build, so upgrading by default would be slower and risks
  breaking the runtime.
* **Demo UI:** `gradio` only when `INSTALL_DEMO_UI = True`.

Deliberately **not** installed yet: `segment-anything`, `SAM 2`, Matcher and
PerSAM requirements. They are added when their phases begin, so the Phase 0
environment stays small and fast to verify.

Installation is skipped for any package whose installed version already
satisfies the requirement, which keeps re-runs cheap.

In [ ]:
import importlib.util
import subprocess
import sys

# pip name -> (import name, minimum acceptable version)
REQUIRED_PACKAGES = {
    "numpy": ("numpy", "1.24"),
    "scipy": ("scipy", "1.10"),
    "scikit-learn": ("sklearn", "1.3"),
    "matplotlib": ("matplotlib", "3.7"),
    "pillow": ("PIL", "9.5"),
    "tqdm": ("tqdm", "4.66"),
    "pandas": ("pandas", "2.0"),
    "pyyaml": ("yaml", "6.0"),
    "opencv-python-headless": ("cv2", "4.8"),
}
VISION_PACKAGES = {"torchvision": ("torchvision", "0.17")}
TRANSFORMER_PACKAGES = {"transformers": ("transformers", "4.40")}


def _importable(import_name: str) -> bool:
    try:
        return importlib.util.find_spec(import_name) is not None
    except Exception:
        return False


def _installed_version(distribution: str):
    from importlib import metadata

    try:
        return metadata.version(distribution)
    except Exception:
        return None


def install_packages(spec, label):
    """Install only packages that are missing.

    An already-importable module is never replaced: reinstalling `cv2` over a
    working OpenCV build is a common source of broken Colab runtimes. Version
    floors are therefore reported, not enforced - Section 4 prints what is
    actually importable.
    """
    todo = {name: spec[name] for name in spec if not _importable(spec[name][0])}
    if not todo:
        print(f"[deps] {label}: already available")
        return {}
    print(f"[deps] {label}: installing {', '.join(sorted(todo))}")
    # One pip call keeps the (slow) dependency resolution to a single pass.
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
               *[f"{name}>={minimum}" for name, (_module, minimum) in todo.items()]]
    result = subprocess.run(command, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, check=False)
    output = result.stdout.decode("utf-8", errors="replace").strip()
    if result.returncode != 0:
        print(output[-2000:])
        print(f"! pip failed for {label}; see the output above.")
    return {name: _installed_version(name) for name in todo}


installed = {}
if INSTALL_PACKAGES:
    installed.update(install_packages(REQUIRED_PACKAGES, "core"))
    if INSTALL_TORCHVISION:
        installed.update(install_packages(VISION_PACKAGES, "vision"))
    if INSTALL_TRANSFORMERS:
        installed.update(install_packages(TRANSFORMER_PACKAGES, "backbone"))
    if INSTALL_DEMO_UI:
        installed.update(install_packages({"gradio": ("gradio", "4.0")}, "demo ui"))
if INSTALL_TORCH:
    print("[deps] torch: reinstalling from the CUDA wheel index")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                    "torch", "torchvision"],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)

# Restart required? Imports cached before an install would be stale.
if installed:
    print("[deps] packages changed - restart the runtime "
          "(Runtime > Restart runtime) and re-run this notebook from Section 1.")

In [ ]:
# Imported versions after installation.
from scripts import environment_check as envcheck

versions = envcheck.get_package_versions()
missing_required = [p for p in ("numpy", "scipy", "sklearn", "matplotlib", "PIL",
                                "tqdm", "pandas", "yaml", "cv2") if versions.get(p) is None]
print(f"{'package':<16}{'version':<16}status")
print("-" * 42)
for name in envcheck.TRACKED_PACKAGES:
    version = versions.get(name)
    if name == "torch" and version is None:
        status = "missing (blocking)"
    elif version is None and name in ("torchvision", "transformers"):
        status = "missing (needed from Phase 1)"
    elif version is None and name == "gradio":
        status = "optional (demo phase)"
    else:
        status = "ok" if version else "missing"
    print(f"{name:<16}{str(version or '-'):<16}{status}")

if missing_required:
    print("! Missing Phase 0 packages: " + ", ".join(missing_required))
else:
    print("All Phase 0 packages are available.")

## Section 5 - GPU diagnostics and CUDA test

The runtime is probed at execution time - no GPU model is hard-coded, because
Colab hardware varies (T4, L4, A100, ...).

Reported: GPU name, GPU memory (total and free), CUDA version, PyTorch version,
CUDA availability, compute capability, driver version, and the result of a real
matrix multiplication on the GPU. The runtime is then judged against this
project's requirements (minimum 8 GB VRAM, 15 GB recommended) with actionable
warnings.

In [ ]:
# ---- explicit GPU test: tensor -> CUDA -> matmul -> report ---------------
import time

print(f"GPU name        : {envcheck.get_gpu_info().get('name')}")
gpu = envcheck.get_gpu_info()
print(f"GPU memory      : {gpu.get('total_memory_gb')} GB total / "
      f"{gpu.get('free_memory_gb')} GB free")
cuda = envcheck.get_cuda_info()
print(f"CUDA version    : build {cuda.get('cuda_build_version')} / "
      f"runtime {cuda.get('cuda_runtime_version')}")
print(f"CUDA available  : {cuda.get('cuda_available')}")
print(f"PyTorch version : {envcheck.get_torch_info().get('version')}")

try:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available in this runtime.")

    device = torch.device("cuda")
    size = 1024
    x = torch.randn(size, size, device=device)     # 1) small tensor
    w = torch.randn(size, size, device=device)     # 3) matrix operation
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(10):
        y = x @ w
    torch.cuda.synchronize()
    per_iter = (time.perf_counter() - start) / 10 * 1000

    print(f"\n[gpu-test] device          : {device} ({torch.cuda.get_device_name(0)})")
    print(f"[gpu-test] tensor          : {tuple(x.shape)} on {x.device}")
    print(f"[gpu-test] matmul          : x @ w -> {tuple(y.shape)}")
    print(f"[gpu-test] mean |y|        : {y.abs().mean().item():.4f} "
          f"(expected ~{size})")
    print(f"[gpu-test] time            : {per_iter:.3f} ms / iteration")
    print(f"[gpu-test] peak VRAM       : {torch.cuda.max_memory_allocated(0)/1024**2:.1f} MB")
    print("[gpu-test] RESULT          : SUCCESS")
    del x, w, y
    torch.cuda.empty_cache()
except Exception as exc:
    print(f"[gpu-test] RESULT          : FAILED ({type(exc).__name__}: {exc})")
    print("! Switch to a GPU runtime: Runtime > Change runtime type > GPU.")

In [ ]:
# ---- full diagnostics + sufficiency assessment ---------------------------
from scripts import project_setup as psetup

# Thresholds live in configs/paths.yaml (resources:) so they can be tuned
# without touching code.
RESOURCES = psetup.load_paths_config().get("resources", {})
report = envcheck.get_environment_report(
    matrix_size=1024,
    disk_paths=["/content", PERSISTENT_ROOT],
    thresholds=RESOURCES,
)
print(f"thresholds: {RESOURCES}")
envcheck.print_environment_report(report)

if report["warnings"]:
    print("\nHow to react to the warnings above:")
    print("  * GPU missing      -> Runtime > Change runtime type > Hardware accelerator > GPU")
    print("  * Low free VRAM    -> Runtime > Restart runtime, then re-run this notebook")
    print("  * Low Drive space  -> free up space in MyDrive (results/cache are the largest items)")

envcheck.save_environment_report(
    report, os.path.join(PERSISTENT_ROOT, "results", "environment", "env_report.json")
)
print(f"\nJSON report saved under {PERSISTENT_ROOT}/results/environment/env_report.json")

## Section 6 - Project directories

Creates the full tree on Drive, idempotently - existing directories are left
alone, so re-running never deletes or duplicates anything.

```
<drive>/one-shot-segmentation/
    project/      <- code lives in the git repo (/content/one-shot-segmentation)
    data/{raw,processed,cache,splits}
    checkpoints/
    results/{baseline,robustness,proposed,ablation,final}
    logs/
```

Datasets, checkpoints, feature caches and results all live **on Drive**. Only
code and configs live in the git repository.

In [ ]:
from scripts import project_setup as psetup

# Authoritative paths come from configs/paths.yaml in the repository.
paths = psetup.resolve_paths(
    token_overrides={
        "DRIVE_MOUNT": DRIVE_MOUNT,
        "DRIVE_FOLDER": DRIVE_FOLDER,
        "DRIVE_ROOT": PERSISTENT_ROOT,
    },
)
print(psetup.format_paths_table(paths))

dir_result = psetup.ensure_directories(paths)
print(f"[dirs] drive_ready={dir_result['drive_ready']} "
      f"created={len(dir_result['created'])} already_present={len(dir_result['existing'])}")
for directory in dir_result["created"]:
    print(f"  created   : {directory}")
if dir_result["skipped_due_to_unmounted_drive"]:
    print("  skipped   : Drive is not mounted, so these were not created: "
          + ", ".join(dir_result["skipped_due_to_unmounted_drive"]))

# Make the code packages importable for later phases.
psetup.add_repo_to_sys_path(paths.repo_root)
psetup.set_runtime_env(paths)
print(f"[dirs] HF_HOME  = {os.environ.get('HF_HOME')}")
print(f"[dirs] results groups: "
      + ", ".join(sorted(paths.results_directories().keys())))

In [ ]:
# ---- declared datasets and checkpoints (presence only, nothing downloaded)
print("Datasets (declared for later phases):")
for item in psetup.describe_datasets(paths):
    mark = "present" if item["exists"] else "not downloaded"
    print(f"  [{item['name']:<18}] {mark:<15} {item['path']}")

print("\nCheckpoints (declared for later phases):")
for item in psetup.describe_checkpoints(paths):
    mark = "present" if item["exists"] else "not downloaded"
    print(f"  [{item['name']:<18}] {mark:<15} {item['path']}")

print("\nPhase 0 downloads nothing, so 'not downloaded' is the expected state.")
print("Later phases call psetup.dataset_exists()/checkpoint_exists() first so an "
      "artefact present on Drive is never downloaded twice.")

## Section 7 - Reproducibility

Sets and verifies:

* Python `random`, NumPy, PyTorch (CPU) and all CUDA device seeds;
* `torch.backends.cudnn.deterministic`, `benchmark`, `use_deterministic_algorithms`
  and `CUBLAS_WORKSPACE_CONFIG`;
* an experiment metadata record: UTC timestamp, git commit/branch/dirty flag,
  resolved config, seed, GPU and software versions.

Every later experiment calls `start_experiment(...)` and saves that record next
to its results, so any result in `results/` can be traced back to the exact
commit, seed and configuration that produced it.

In [ ]:
from scripts import reproducibility as repro

seeding = repro.set_seed(RANDOM_SEED, deterministic=DETERMINISTIC)
print(f"[repro] CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')}")

# Verification: two identically seeded draws must be identical.
import numpy as np

repro.set_seed(RANDOM_SEED, deterministic=DETERMINISTIC, quiet=True)
first_draw = (np.random.rand(4), np.random.randint(0, 1000, 4))
repro.set_seed(RANDOM_SEED, deterministic=DETERMINISTIC, quiet=True)
second_draw = (np.random.rand(4), np.random.randint(0, 1000, 4))

numpy_ok = np.allclose(first_draw[0], second_draw[0]) and np.array_equal(first_draw[1], second_draw[1])
print(f"[repro] numpy reproducible   : {numpy_ok}")

torch_ok = None
try:
    import torch

    repro.set_seed(RANDOM_SEED, quiet=True)
    a = torch.randn(4, 4)
    repro.set_seed(RANDOM_SEED, quiet=True)
    b = torch.randn(4, 4)
    torch_ok = bool(torch.equal(a, b))
    print(f"[repro] torch reproducible  : {torch_ok}")
except Exception as exc:
    print(f"[repro] torch check skipped : {exc}")

In [ ]:
# ---- experiment metadata ------------------------------------------------
experiment = repro.start_experiment(
    experiment_name="phase0_environment_setup",
    config={
        "notebook": "01_colab_setup.ipynb",
        "paths": paths.to_dict(),
        "seeding": seeding,
        "environment_status": report["summary"]["status"],
    },
    seed=RANDOM_SEED,
    deterministic=DETERMINISTIC,
    repo_root=paths.repo_root,
    notes="Phase 0 setup verification run.",
)
print(experiment.summary())

env_dir = os.path.join(PERSISTENT_ROOT, "results", "environment")
metadata_path = experiment.save(env_dir)
print(f"[repro] metadata -> {metadata_path}")

# Software + git snapshot, handy when writing the thesis appendix.
versions = repro.get_software_versions()
print("\nsoftware versions recorded:")
for key, value in versions.items():
    print(f"  {key:<16} {value}")
print("\ngit info recorded:")
for key, value in repro.get_git_info(paths.repo_root).items():
    print(f"  {key:<16} {value}")

## Section 8 - Feature-cache architecture

DINOv2 features will be reused across every perturbation sweep and every
matching experiment, so they are cached once and reloaded afterwards.

```
image â”€â”€â–º DINOv2 â”€â”€â–º feature tensor â”€â”€â–º data/cache/<model>/<tag>/<split>/<image_id>__<key>.pt
                                                          â””â”€â”€ future experiments just load it
```

The cache filename is built from everything that can change a tensor:

* image identity (`image_id`, `split`, `transform_tag` for perturbed copies),
* model family / tag / config (hidden size, patch size, chosen layers, ...),
* preprocessing (resize, crop, normalisation).

`cache_utils.get_or_load()` is the only entry point feature consumers need: it
calls the extractor on a miss and never on a hit. **No DINOv2 model is loaded in
this section** - the contract is demonstrated with dummy tensors so the storage
format is fixed before any model code exists.

In [ ]:
from scripts import cache_utils as cutils

# Naming contract (demonstrated with dummy image ids).
identity = cutils.image_identity(
    image_path="/data/raw/lvis_v1/val/000000042.jpg",
    image_id="lvis_v1/val/000000042",
    split="val",
    transform_tag="azimuth+30_elev-10_occ0.0",   # perturbed variant
)
model_config = {"hidden_size": 768, "patch_size": 14, "layers": [-1]}
preprocess = {"resize": 518, "crop": None, "normalise": "imagenet"}

key = cutils.build_cache_key(identity, model_tag="dinov2-base",
                             model_config=model_config, preprocess=preprocess)
demo_root = paths.cache_dir_for("dinov2", "dinov2-base")
print(f"[cache] image identity : {identity}")
print(f"[cache] config hash    : {cutils.cache_signature(model_tag='dinov2-base', model_config=model_config, preprocess=preprocess)}")
print(f"[cache] example key    : {key}")
print(f"[cache] example path   : {os.path.join(demo_root, 'val', key + '.pt')}")
print("[cache] a different model tag, resolution or perturbation yields a different "
      "key, so stale features can never be reused silently.")

In [ ]:
# ---- end-to-end demo with dummy tensors (no DINOv2 involved) ------------
cache_demo = cutils.demo_feature_cache(
    os.path.join(paths.cache_root, "demo"), verbose=True
)

print("\nfiles written to the cache:")
for file_path in cache_demo["files"]:
    print(f"  {os.path.relpath(file_path, paths.cache_root)}  "
          f"({os.path.getsize(file_path)/1024:.1f} KiB)")

assert cache_demo["warm_pass_hits"] == cache_demo["status"]["total"], "cache must hit on the second pass"
assert cache_demo["reload_identical"], "cached values must survive a reload"
assert cache_demo["config_isolation_verified"], "a new model config must miss the cache"
print("\n[cache] demo assertions passed: hits on reuse, isolation per config, "
      "identical values after reload.")

## Section 9 - Environment verification

Final gate: everything from the previous sections is re-checked in one place and
a single verdict is printed.

* `PASS` - the environment is ready for Phase 1.
* `WARN` - usable, but read the warnings (typically a small GPU or a package
  that belongs to a later phase).
* `FAIL` - fix the runtime before continuing.

The verdict, the checks and the metadata record are written to
`results/environment/` on Drive so the exact environment behind any later result
can be recovered.

In [ ]:
# ---- Phase 0 verification -------------------------------------------------
checks = []

def add_check(name, ok, detail, blocking=True):
    status = "pass" if ok else ("warn" if not blocking else "fail")
    checks.append({"check": name, "status": status, "detail": str(detail)})
    return status

add_check("GPU detected", bool(report["gpu"]["available"]), report["gpu"]["name"])
add_check("CUDA available", bool(report["cuda"]["cuda_available"]),
          f"CUDA {report['cuda']['cuda_build_version']}")
add_check("GPU inference test", bool(report["gpu_test"]["success"]),
          report["gpu_test"].get("error") or f"{report['gpu_test'].get('ms_per_iteration')} ms/iter")
add_check("Memory sufficient", report["memory"]["overall"] in ("ok", "degraded"),
          report["memory"]["overall"], blocking=False)
add_check("Drive mounted", drive_status["available"], drive_status["message"], blocking=False)
add_check("Repository available", os.path.isdir(PROJECT_ROOT), PROJECT_ROOT)
add_check("Project files present", not missing, f"{len(REQUIRED_FILES) - len(missing)}/{len(REQUIRED_FILES)}")
add_check("Directories created", not dir_result["skipped_due_to_unmounted_drive"],
          f"{len(paths.all_directories())} persistent directories")
add_check("Phase 0 packages", not missing_required,
          "all present" if not missing_required else f"missing {missing_required}")
add_check("Seeding reproducible", bool(numpy_ok) and torch_ok is not False,
          f"numpy={numpy_ok}, torch={torch_ok}", blocking=False)
add_check("Deterministic kernels", DETERMINISTIC, seeding.get("cudnn_deterministic"), blocking=False)
add_check("Feature cache contract", bool(cache_demo["reload_identical"]),
          f"{cache_demo['status']['hits']}/{cache_demo['status']['total']} hits on reuse")
add_check("Experiment metadata", os.path.isfile(metadata_path), os.path.basename(metadata_path))

statuses = {c["status"] for c in checks}
verdict = "FAIL" if "fail" in statuses else ("WARN" if "warn" in statuses else "PASS")

print(f"{'CHECK':<32}{'STATUS':<8}DETAIL")
print("-" * 96)
for check in checks:
    print(f"{check['check']:<32}{check['status'].upper():<8}{check['detail']}")
print("-" * 96)
print(f"PHASE 0 VERDICT: {verdict}")
if report["warnings"]:
    print("\nopen warnings:")
    for warning in report["warnings"]:
        print(f"  ! {warning}")

In [ ]:
# ---- persist the verification record ------------------------------------
verification = {
    "phase": "0 - Colab environment setup",
    "verdict": verdict,
    "checks": checks,
    "warnings": report["warnings"],
    "config": CONFIG,
    "paths": paths.to_dict(),
    "drive": drive_status,
    "gpu": report["gpu"],
    "cuda": report["cuda"],
    "torch": report["torch"],
    "package_versions": versions,
    "environment_report": report,
    "cache_demo": {k: v for k, v in cache_demo.items() if k != "files"},
    "metadata_file": metadata_path,
}

import json

env_dir = os.path.join(PERSISTENT_ROOT, "results", "environment")
os.makedirs(env_dir, exist_ok=True)
verification_path = os.path.join(env_dir, "phase0_verification.json")
with open(verification_path, "w", encoding="utf-8") as handle:
    json.dump(verification, handle, indent=2, default=str)

print(f"[verify] {verification_path}")
print("\nWhere things now live (Google Drive):")
print(f"  datasets       : {paths.raw_root}")
print(f"  checkpoints    : {paths.checkpoints_root}")
print(f"  feature cache  : {paths.cache_root}")
print(f"  results        : {paths.results_root}")
print(f"  logs           : {paths.logs_root}")

PHASE0_OK = verdict != "FAIL"
print(f"\nPHASE0_OK = {PHASE0_OK}")

## Section 10 - Next steps and phase boundaries

Phase 0 is complete once `PHASE0_OK` is `True`. Stop here and confirm the
environment before starting Phase 1.

**Deliberately not done in Phase 0** - no dataset downloads, no DINOv2 feature
extraction, no SAM / SAM2, no Matcher, no PerSAM, no perturbations, no
robustness experiments, no proposed method.

### Where to look first

| Item | Location |
| --- | --- |
| Datasets (raw / processed) | `<drive>/one-shot-segmentation/data/raw`, `data/processed` |
| Dataset splits | `<drive>/one-shot-segmentation/data/splits` |
| Model checkpoints | `<drive>/one-shot-segmentation/checkpoints` |
| Cached DINOv2 features | `<drive>/one-shot-segmentation/data/cache/dinov2/<model_tag>` |
| Experiment results | `<drive>/one-shot-segmentation/results/{baseline,robustness,proposed,ablation,final}` |
| Environment reports / metadata | `<drive>/one-shot-segmentation/results/environment` |
| Logs | `<drive>/one-shot-segmentation/logs` |
| Code and configs | git repo, cloned at `/content/one-shot-segmentation` |

### Re-running later

1. `Runtime > Change runtime type > GPU`, then `Runtime > Restart runtime`.
2. Re-open this notebook and run all cells (`Runtime > Run all`).
   Clones, satisfied packages, existing directories and cached features are
   skipped automatically.

### Phase 1 (not started)

Download/verify datasets into `data/raw`, build deterministic splits into
`data/splits`, then extract DINOv2 features with the Section 8 cache
(`cache_utils.get_or_load`) and record every run with
`reproducibility.start_experiment`. Add Matcher dependencies at that point.